# Image feature extraction: resnet18

In [9]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:x
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
else:
    PROJECT_ROOT = Path('../../..').resolve()


DATA_ROOT = PROJECT_ROOT / 'CGMacros'  # raw participant folders + photos/ live here
PROCESSED_IMAGE_DIR = PROJECT_ROOT / 'processed_data' / 'img_preprocessing'
PROCESSED_CGM_DIR = PROJECT_ROOT / 'processed_data' / 'processed_5min'
sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}")
print(f"REPO_ROOT={PROJECT_ROOT}")
print(f"DATA_ROOT={DATA_ROOT}")
print(f"PROCESSED_IMAGE_DIR={PROCESSED_IMAGE_DIR}")
print(f"PROCESSED_CGM_DIR={PROCESSED_CGM_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
IN_COLAB=True
REPO_ROOT=/content/drive/MyDrive/dissertation
DATA_ROOT=/content/drive/MyDrive/dissertation/CGMacros
PROCESSED_IMAGE_DIR=/content/drive/MyDrive/dissertation/processed_data/img_preprocessing
PROCESSED_CGM_DIR=/content/drive/MyDrive/dissertation/processed_data/processed_5min


In [8]:
import json
import pickle
import time

import numpy as np
import torch
from PIL import Image
from tqdm.auto import tqdm

from src.models.image_encoders import load_encoder, IMG_TRANSFORM

ENCODER_NAME = 'resnet18'
BATCH_SIZE = 64

DEVICE = ('cuda' if torch.cuda.is_available()
          else 'mps' if torch.backends.mps.is_available()
          else 'cpu')
print(f'device: {DEVICE}')

FEATURES_DIR = PROCESSED_IMAGE_DIR / 'img_features'
FEATURES_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = FEATURES_DIR / f'img_features_{ENCODER_NAME}.pkl'

device: cuda


## Load the verified image list

In [6]:
resolved_keys_path = PROCESSED_IMAGE_DIR / 'resolved_image_keys.json'
with open(resolved_keys_path) as f:
    resolved_keys = json.load(f)

image_paths = {}
for key in resolved_keys:
    participant_id, filename = key.split('/', 1)
    image_paths[key] = DATA_ROOT / participant_id / 'photos' / filename

print(f'{len(image_paths)} verified images to encode')

1517 verified images to encode


## Extract resnet18 features

In [7]:
model, feat_dim, forward_fn = load_encoder(ENCODER_NAME, device=DEVICE)
print(f'{ENCODER_NAME}  feat_dim={feat_dim}')

keys = list(image_paths.keys())
features = {}
skipped = []

t0 = time.time()
pbar = tqdm(range(0, len(keys), BATCH_SIZE), desc=ENCODER_NAME)
for i in pbar:
    batch_keys = keys[i: i + BATCH_SIZE]
    imgs, valid_keys = [], []
    for k in batch_keys:
        try:
            img = Image.open(image_paths[k]).convert('RGB')
            imgs.append(IMG_TRANSFORM(img))
            valid_keys.append(k)
        except Exception as e:
            skipped.append((k, str(e)))

    if not imgs:
        continue

    batch_tensor = torch.stack(imgs).to(DEVICE)
    with torch.no_grad():
        feats = forward_fn(model, batch_tensor).cpu().numpy()

    for k, feat in zip(valid_keys, feats):
        features[k] = feat.astype(np.float32)

    pbar.set_postfix(encoded=len(features), skipped=len(skipped))

elapsed = time.time() - t0
del model
if DEVICE == 'cuda':
    torch.cuda.empty_cache()

print(f'\nencoded {len(features)}/{len(keys)} images in {elapsed:.0f}s')
if skipped:
    print(f'\u26a0\ufe0f UNEXPECTED: {len(skipped)} keys failed despite being pre-verified -- '
          f'investigate, do not ignore:')
    print('  ', skipped[:5], '...' if len(skipped) > 5 else '')
else:
    print('0 skipped, as expected (all keys were pre-verified by images_preprocessing.ipynb)')

# Sample output: one concrete vector, not just a count.
_demo_key = keys[0]
_demo_vec = features[_demo_key]
print(f"\nsample output -> key='{_demo_key}'")
print(f'  shape={_demo_vec.shape}  dtype={_demo_vec.dtype}  '
      f'first5={np.round(_demo_vec[:5], 3)}  L2norm={np.linalg.norm(_demo_vec):.2f}')

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 238MB/s]


resnet18  feat_dim=512


resnet18:   0%|          | 0/24 [00:00<?, ?it/s]


encoded 1517/1517 images in 1333s
0 skipped, as expected (all keys were pre-verified by images_preprocessing.ipynb)

sample output -> key='CGMacros-001/00000005-PHOTO-2020-5-1-14-23-0.jpg'
  shape=(512,)  dtype=float32  first5=[1.527 0.202 0.839 1.038 0.244]  L2norm=29.43


## Save

In [10]:
with open(OUT_PATH, 'wb') as f:
    pickle.dump(features, f)

print(f'saved {len(features)} x {feat_dim}-d {ENCODER_NAME} features to {OUT_PATH} '
      f'({OUT_PATH.stat().st_size / 1e6:.1f} MB)')

assert len(features) == len(resolved_keys), (
    f'expected {len(resolved_keys)} features, got {len(features)} -- investigate before using '
    f'this file downstream'
)
print('count matches resolved_image_keys.json exactly -- OK')

saved 1517 x 512-d resnet18 features to /content/drive/MyDrive/dissertation/processed_data/img_preprocessing/img_features/img_features_resnet18.pkl (3.2 MB)
count matches resolved_image_keys.json exactly -- OK
